# NOVA SN68 — scoring Boltz-2 sur GPU Kaggle

Note un lot de molécules avec Boltz-2 **exactement comme les validateurs NOVA** (même entrée, même MSA, mêmes paramètres, même graine) et sauvegarde les scores dans `/kaggle/working/`.

**Avant de lancer :**
1. Paramètres du notebook → *Accelerator* : **GPU T4 x2** (ou P100), *Internet* : **On** (compte vérifié par téléphone requis).
2. Pour laisser tourner sans garder l'onglet ouvert : **Save Version → Save & Run All (Commit)**, puis récupérer le CSV dans l'onglet *Output* de la version.

Usage de recherche uniquement : ce notebook ne soumet rien au réseau Bittensor.

In [ ]:
# Paramètres
BRANCH = "claude/eager-mayer-ot25x6"
BATCH = "inputs/kaggle_batch.csv"   # CSV avec une colonne 'name' (+ colonnes libres conservées)
TOP = None                          # None = tout le lot ; sinon les N premières lignes
RUN = "kaggle_batch1"
MULTI_GPU = True                    # False si le mode 2 GPU pose problème

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q --depth 1 -b {BRANCH} https://github.com/0xRozier/Nova.git /kaggle/working/Nova
%cd /kaggle/working/Nova
!mkdir -p data && wget -q -O data/molecules.sqlite "https://huggingface.co/datasets/Metanova/Mol-Rxn-DB/resolve/main/molecules.sqlite"
!ls -la data

## Environnement isolé pour Boltz 2.2.1
Boltz 2.2.1 exige `numpy<2` et Python ≤ 3.12 : on l'installe dans un venv séparé pour ne pas casser l'image Kaggle (~5 min).

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 /kaggle/working/bvenv
!uv pip install -q --python /kaggle/working/bvenv/bin/python torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126
!uv pip install -q --python /kaggle/working/bvenv/bin/python boltz==2.2.1 "pyarrow<17" pandas requests pyyaml
!/kaggle/working/bvenv/bin/python -c "import torch, boltz; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))"

## Précision de calcul
Les validateurs tournent en `bf16-mixed` sur des GPU récents (A100…). Les T4/P100 de Kaggle (architecture < 8.0) ne gèrent pas le bf16 nativement : dans ce cas on passe Boltz en précision 32 bits. C'est un léger écart, mesuré par les molécules de calibration du lot.

In [ ]:
import subprocess, re, pathlib
py = "/kaggle/working/bvenv/bin/python"
cap = subprocess.check_output([py, "-c", "import torch; print('%d.%d' % torch.cuda.get_device_capability(0))"], text=True).strip()
n_gpus = int(subprocess.check_output([py, "-c", "import torch; print(torch.cuda.device_count())"], text=True))
main_py = pathlib.Path(subprocess.check_output([py, "-c", "import boltz.main as m; print(m.__file__)"], text=True).strip())
print("compute capability", cap, "| GPUs", n_gpus)
if float(cap) < 8.0:
    src = main_py.read_text()
    if '"bf16-mixed"' in src:
        main_py.write_text(src.replace('"bf16-mixed"', '32'))
    print("bf16 non supporté -> précision 32 bits")
else:
    print("bf16 supporté -> réglage identique aux validateurs")
EXTRA = f"--no_kernels --devices {n_gpus if MULTI_GPU else 1}"

## Scoring
Le premier lancement télécharge les poids Boltz-2 (~6 Go). Si la session s'interrompt, relancer cette cellule avec le même `RUN` reprend là où elle s'était arrêtée.

In [ ]:
top = f"--top {TOP}" if TOP else ""
!{py} -m scripts.boltz_score --file {BATCH} {top} --accelerator gpu --run {RUN} --boltz /kaggle/working/bvenv/bin/boltz --extra="{EXTRA}"

In [ ]:
import shutil, pandas as pd
out = f"/kaggle/working/scores_{RUN}.csv"
shutil.copy(f"data/boltz/{RUN}/scores.csv", out)
df = pd.read_csv(out)
print(out, len(df), "molécules")
df.head(60)